# ĐỒ ÁN MÔN HỌC — NT120: NHẬP MÔN TRÍ TUỆ NHÂN TẠO CHO MẠNG MÁY TÍNH VÀ AN TOÀN THÔNG TIN

Coursework notebook explaining and validating the synthetic flow pipeline; real-data training remains blocked.
Notebook đồ án giải thích và kiểm pipeline flow synthetic; training dữ liệu thật vẫn bị chặn.

## Phát hiện DDoS trong mạng IoT bằng Flow-Based CNN, hướng đến Swarm Learning trên thiết bị biên

| Thông tin | Nội dung do nhóm tự điền |
|---|---|
| Lớp | [TỰ ĐIỀN] |
| Nhóm số | [TỰ ĐIỀN; đổi NhomXX trong tên file] |
| Giảng viên | [TỰ ĐIỀN] |
| Học kỳ / Ngày nộp | [TỰ ĐIỀN] |

| MSSV | Họ tên | Vai trò / đóng góp | Email |
|---|---|---|---|
| [TỰ ĐIỀN] | [TỰ ĐIỀN] | [TỰ ĐIỀN] | [TỰ ĐIỀN] |

**Bản nháp kỹ thuật, chưa phải kết quả huấn luyện Edge-IIoTset hoàn chỉnh.**
Notebook triển khai và kiểm chứng luồng synthetic bằng source dự án. Các mục
training/evaluation thật vẫn BLOCKED vì thiếu packet/flow ground truth verified.
Không thay số liệu còn thiếu bằng kết quả synthetic hoặc kết quả của bài báo.


## Cách sử dụng notebook này

Giữ nguyên 13 mục lớn theo khung NT120. Tên file hiện có `NhomXX` để người dùng
tự điền thông tin cá nhân. Dùng **Restart & Run All**; các cell cần state từ cell
trước, không chạy ngẫu nhiên rồi dùng kết quả cũ.

Notebook có hai loại kết quả:

1. **Software/synthetic validation:** kiểm ordering, packet-to-flow, padding,
   HDF5 và CNN forward khi môi trường tương thích. Đây là công việc có thể chạy.
2. **Nghiên cứu dữ liệu thật:** provenance, nhãn, split, train, validation và
   final test. Những phần chưa được duyệt in trạng thái BLOCKED và không chạy fit.

Một lần Run All không có exception **không đồng nghĩa** mọi gate đã PASS.
Cell tổng kết liệt kê PASS/BLOCKED/SKIPPED riêng; missing source/dependencies
không được chuyển thành thành công. Bản nộp cuối cần giải quyết các blocker.

### Chuẩn bị source và môi trường

- Local: mở notebook trong workspace `nids-ml-pipeline`, hoặc đặt biến môi trường
  `NIDS_WORKSPACE` đến đúng root chứa `src/`, `configs/`, `contracts/`.
- Colab: đưa **source** từ repo GitHub của nhóm về `/content/nids-ml-pipeline`
  sau khi repo được công bố. Cần pin commit khi chạy thí nghiệm. Repo hiện chưa
  được notebook tự giả định URL; không tự clone, tải dataset hoặc đọc token.
- Chỉ upload `.ipynb` sẽ chưa có package `nids_ml`; các bước cần package sẽ báo
  `BLOCKED_SOURCE_NOT_AVAILABLE`. Không dùng code mẫu như kết quả đã chạy.
- Kernel chạy phần stdlib; `NIDS_PYTHON` có thể chỉ định một interpreter Python
  3.11 đã được operator chuẩn bị cho HDF5/CNN. Nếu không đặt, dùng kernel hiện tại.
- HDF5 yêu cầu Python 3.11 / NumPy 1.26.4 / h5py 3.16.0. CNN yêu cầu thêm
  TensorFlow 2.15.1 / Keras 2.15.0. Notebook **không tự cài/downgrade** thư viện.
  Colab managed runtime có thể dùng version khác; hiện chưa có live Colab PASS.
- Có thể chuẩn bị venv Python 3.11 riêng và cài extra HDF5 bằng
  `python -m pip install --editable '.[hdf5]'` trong môi trường được duyệt.
  Extra này không tự cài TensorFlow. Runtime CNN cần kiểm pins/ABI riêng.

Không Docker, SSH, deploy, Git commit/push hoặc kết nối hai board từ notebook.
Khi chạy demo có source, chỉ tạo run mới dưới `artifacts/runs/`; không ghi đè
run cũ, không đọc CSV/PCAP thật và không tự cleanup artifacts đã tồn tại.


## Mục lục

1. Giới thiệu và phát biểu bài toán
2. Khảo sát các giải pháp liên quan
3. Thiết lập môi trường và nạp dữ liệu
4. Tiền xử lý dữ liệu
5. Phương pháp tiếp cận
6. Thực nghiệm và đánh giá kết quả
7. Phân tích và thảo luận
8. Kết luận và hướng phát triển
9. Đóng góp của các thành viên
10. Tài liệu tham khảo
11. Phụ lục (tùy chọn)
12. Checklist tự đánh giá trước khi nộp
13. Gợi ý chuẩn bị phần trình bày trước lớp

## 1. Giới thiệu và phát biểu bài toán

### 1.1 Bối cảnh và động lực

Thiết bị IoT/IIoT tạo lưu lượng đa dạng và có tài nguyên giới hạn. DDoS làm suy
giảm tính sẵn sàng bằng cách gây quá tải tài nguyên mạng hoặc dịch vụ. Mô hình
phát hiện cần nhận diện patterns trong traffic và được đánh giá với nhãn đáng
tin cậy, thay vì chỉ nhận ra IP/scenario của bộ dữ liệu.

Nhánh Flow-based biểu diễn một flow window bằng chuỗi đặc trưng packet, giúp
CNN khai thác cấu trúc theo thứ tự thời gian. Mục tiêu dài hạn là mỗi edge node
giữ dữ liệu local, huấn luyện và trao đổi trọng số qua Independent OpenSwarm.
Trong notebook này chưa có CNN Swarm training; kiểm thử hạ tầng tiny-model trên
hai board trước đây là một loại bằng chứng khác.


### 1.2 Phát biểu bài toán

- **Input nghiên cứu:** PCAP Edge-IIoTset, metadata capture và ground truth;
  packet IPv4 TCP/UDP/ICMP đủ eligibility được nhóm theo bidirectional 5-tuple.
- **Input CNN:** float32 `(batch, n, 11, 1)`, với n=20 là profile chính và n=10
  dùng đối chiếu Table 3; window 10 giây, zero padding cho phần thiếu packet.
- **Output mục tiêu:** xác suất binary DDoS, positive class=1, verified benign=0.
  UNKNOWN/MIXED không được tự biến thành 0 hoặc đưa vào supervised loss.
- **Input demo hiện tại:** metadata synthetic nhỏ, không phải PCAP/raw dataset.
- **Giả định cần kiểm:** packet chronology, capture-local flow identity, labels
  theo evidence và split không trùng capture/flow giữa train/validation/test.

Không coi tất cả attack types trong Edge-IIoTset là DDoS. IP/ports dùng định danh
flow ngoài tensor; labels, capture ID và source ordinal không phải CNN features.


### 1.3 Mục tiêu cụ thể

1. Tái lập được pipeline synthetic: source hash → stable processing view →
   flow tensor n20/n10 → lưu/đọc HDF5; xác minh shape/dtype/lineage bằng assertions.
2. Đối chiếu CNN 2.241 tham số với Table 3 và giải thích n10/n20 khác nhau.
3. Bảo đảm UNKNOWN/MIXED bị loại khỏi supervised training; không dùng majority
   voting hoặc filename làm packet ground truth.
4. Khi đủ annotation: split theo đơn vị chống leakage, train baseline tập trung,
   đo precision/recall/F1/ROC-AUC và so sánh baseline trên cùng test protocol.
5. Sau baseline: tích hợp CNN vào hai edge nodes và đo convergence, communication
   cùng CPU/RAM. Các mục 4–5 chưa có kết quả trong bản nháp này.

Không đặt mục tiêu accuracy tùy ý rồi trình bày như một kết quả đã đạt.


### 1.4 Phạm vi và giới hạn

Chỉ nghiên cứu nhánh Flow CNN. Kiến trúc tham khảo bài multimodal [1], nhưng
không triển khai nhánh Packet CNN/concatenation/blockchain nguyên bản. Lõi
Independent SN/SL/User-TF được phát triển độc lập, không dùng source image cũ
chưa được cấp phép. Hai board khác setup 3–5 nodes của bài báo.

Notebook không cài TShark, giải mã PCAP thật, thu traffic, tạo attack traffic,
chia dataset thật hoặc chạy training. Synthetic data minh họa phần mềm; không
thể dùng nó để kết luận độ chính xác phát hiện DDoS trong thực tế.


## 2. Khảo sát các giải pháp liên quan

| Nguồn | Ý tưởng chính | Dự án kế thừa / khác biệt |
|---|---|---|
| Bài multimodal Swarm Learning [1] | Kết hợp Flow/Packet branches và học phân tán | Chọn Flow-only; chưa tuyên bố exact reproduction |
| LUCID [2] | Packet trace → ma trận flow, CNN nhỏ | Đối chiếu 11 features, window, normalization và CNN; thêm lineage/fail-closed |
| Edge-IIoTset [3] | Traffic IoT/IIoT, raw captures và curated CSV | Dữ liệu local đã có; row labels chưa tự thành verified packet labels |
| TensorFlow/Keras và h5py [4–5] | Framework CNN và lưu numeric tensors | Pin runtime đang kiểm; notebook gọi source thay vì copy thuật toán |

Table 3 của [1] mô tả input n10, Conv output `(8,1,64)`; thí nghiệm sensitivity
dùng n20,t10s. Cùng kernel `(3,11)` với n20 cho `(18,1,64)`, không phải 8.
Kernel/stride/padding được đối chiếu với source reference, không đều được công
bố trực tiếp trong bảng. Nhóm ghi đây là adapted implementation và công bố
những giả định; F1 của tác giả không được ghi làm F1 của nhóm.


## 3. Thiết lập môi trường và nạp dữ liệu

### 3.1 Cài đặt và nạp thư viện

**Cell tiếp theo làm gì?** Khởi tạo biến cấu hình notebook và seed 42; chỉ import
stdlib để cell vẫn hoạt động khi Colab chưa có đúng HDF5/TensorFlow. Các cells
sau mới quyết định module nào đủ runtime để chạy.

**Đầu vào:** biến môi trường do operator đặt nếu cần (`NIDS_WORKSPACE`,
`NIDS_PYTHON`). **Đầu ra:** cấu hình, bảng trạng thái `RESULTS` và helper in JSON.
Không có cài package, tải dữ liệu, train hoặc sửa random state ngoài process.
Seed hỗ trợ tái lập; không hứa bit-for-bit giữa CPU/framework versions khác nhau.


In [ ]:
# Cấu hình chung; thông tin nhóm để người dùng tự điền trong Markdown.
import collections
import datetime as dt
import hashlib
import importlib.util
import json
import math
import os
from pathlib import Path
import platform
import random
import shutil
import struct
import subprocess
import sys
import uuid

SEED = 42
random.seed(SEED)
WORKSPACE_HINT = os.environ.get("NIDS_WORKSPACE", "").strip()
PROJECT_PYTHON = os.environ.get("NIDS_PYTHON", sys.executable)
ENABLE_CNN_FORWARD = True  # Chỉ khởi tạo/forward khi đúng runtime; không fit.
RESULTS = {"real_training": "BLOCKED_GROUND_TRUTH", "physical_swarm": "NOT_RUN_IN_NOTEBOOK"}
WORKSPACE = None
RUN_DIR = None
runtime = {}
HDF5_READY = CNN_READY = False
hdf5_reports = {}
ordered = {}

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False, default=str))

show({"seed": SEED, "kernel_python": platform.python_version(),
      "kernel_machine": platform.machine(), "automatic_install": False,
      "scope": "SYNTHETIC_DEMO; real training blocked"})


#### 3.1.1 Xác định đúng source workspace

Cell kiểm root chứa cả package và fixture, rồi thêm `src/` vào import path.
Nếu đã đặt `NIDS_WORKSPACE`, dùng đúng path đó; không tự chuyển sang source
khác khi path sai. Nếu chưa đặt, tìm trong current directory/parents và các vị
trí Colab phổ biến, không quét ổ đĩa. Không clone hoặc sửa repo nào.

Khi thiếu source, các bước phụ thuộc in BLOCKED. Cần đưa source lên runtime
trước khi coi đây là notebook chạy được đầy đủ. `source_fingerprint` bên dưới
chỉ hash source/config/schema để trace code đã dùng, không phải Git commit,
không phải checksum dataset và không là bằng chứng license/nguồn đáng tin.


In [ ]:
def valid_workspace(path):
    return ((path / "src/nids_ml/flows/ordered.py").is_file()
            and (path / "configs/packet-ordering-synthetic.example.json").is_file()
            and (path / "pyproject.toml").is_file())

if WORKSPACE_HINT:
    candidates = [Path(WORKSPACE_HINT).expanduser().resolve()]
else:
    parents = [Path.cwd(), *Path.cwd().parents]
    candidates = [p for parent in parents for p in (parent, parent / "nids-ml-pipeline")]
    candidates += [Path("/content/nids-ml-pipeline"), Path("/content/openswarm/nids-ml-pipeline")]

WORKSPACE = next((p.resolve() for p in candidates if valid_workspace(p)), None)
if WORKSPACE is None:
    RESULTS["source"] = "BLOCKED_SOURCE_NOT_AVAILABLE"
    show({"status": RESULTS["source"], "action": "Đưa source lên runtime hoặc đặt NIDS_WORKSPACE rồi Run All."})
else:
    sys.path.insert(0, str(WORKSPACE / "src"))
    source_hash = hashlib.sha256()
    for folder in ("src", "configs", "contracts"):
        for path in sorted((WORKSPACE / folder).rglob("*")):
            if path.is_file() and path.suffix in (".py", ".json"):
                source_hash.update(path.relative_to(WORKSPACE).as_posix().encode() + b"\0")
                source_hash.update(hashlib.sha256(path.read_bytes()).digest())
    RESULTS["source"] = "PASS_SOURCE_LOCATED"
    show({"status": RESULTS["source"], "workspace": WORKSPACE,
          "source_fingerprint": source_hash.hexdigest(), "git_commit": "NOT_QUERIED"})


#### 3.1.2 Kiểm runtime và thiết lập worker CPU

Cell dùng interpreter `PROJECT_PYTHON` để đọc versions qua metadata, chưa import
TensorFlow. HDF5/CNN chỉ chạy nếu phiên bản khớp contract. Python của notebook
kernel và worker có thể khác nhau; kernel chỉ orchestration/hiển thị, worker
thực thi phần native libraries. Đổi `NIDS_PYTHON` phải thực hiện trước Run All.

Helper `run_project` chạy Python trực tiếp bằng argument list, không shell,
giới hạn 120 giây/lệnh và threads=1. Không đặt GPU hay tăng workload ngầm.
Sai runtime có trạng thái BLOCKED; khi runtime đã phù hợp nhưng app có lỗi,
exception vẫn hiện để người dùng xử lý, không che bằng PASS. Stderr framework
có thể chứa cảnh báo như known GCS duplicate `gs`; nó không chứng minh remote
GCS hoạt động. Chỉ dùng synthetic local data, không truy cập GCS.


In [ ]:
def run_project(arguments, payload=None, timeout=120):
    arguments = list(arguments)
    if WORKSPACE is not None and arguments and arguments[0] == "-c":
        # Embedded/isolated Python có thể bỏ qua PYTHONPATH; thêm src tường minh.
        arguments[1] = "import sys\nsys.path.insert(0, " + repr(str(WORKSPACE / "src")) + ")\n" + arguments[1]
    env = dict(os.environ)
    env.update(PYTHONDONTWRITEBYTECODE="1", OMP_NUM_THREADS="1", TF_NUM_INTRAOP_THREADS="1",
               TF_NUM_INTEROP_THREADS="1", CUDA_VISIBLE_DEVICES="-1", TF_CPP_MIN_LOG_LEVEL="2")
    if WORKSPACE is not None:
        env["PYTHONPATH"] = str(WORKSPACE / "src") + os.pathsep + env.get("PYTHONPATH", "")
    result = subprocess.run([PROJECT_PYTHON, "-B", *arguments], cwd=WORKSPACE,
        input=None if payload is None else json.dumps(payload), capture_output=True,
        text=True, encoding="utf-8", env=env, timeout=timeout)
    if result.returncode != 0:
        raise RuntimeError(f"Project worker failed ({result.returncode}): {result.stderr[-2500:]} {result.stdout[-1500:]}")
    value = json.loads(result.stdout)
    if result.stderr.strip():
        print("Worker stderr (bounded):", result.stderr[-1200:])
    return value

probe = r"""
import importlib.metadata as metadata, json, platform, sys
versions = {}
for name in ('numpy', 'h5py', 'tensorflow', 'keras'):
    try: versions[name] = metadata.version(name)
    except metadata.PackageNotFoundError: versions[name] = 'NOT_INSTALLED'
print(json.dumps(dict(python=list(sys.version_info[:2]), python_text=platform.python_version(),
                     machine=platform.machine(), packages=versions)))
"""
if shutil.which(PROJECT_PYTHON) is None and not Path(PROJECT_PYTHON).is_file():
    RESULTS["runtime"] = "BLOCKED_INTERPRETER_NOT_FOUND"
    show({"status": RESULTS["runtime"]})
else:
    runtime = run_project(["-c", probe])
    versions = runtime["packages"]
    HDF5_READY = runtime["python"] == [3, 11] and versions["numpy"] == "1.26.4" and versions["h5py"] == "3.16.0"
    CNN_READY = runtime["python"] == [3, 11] and versions["numpy"] == "1.26.4" and versions["tensorflow"] == "2.15.1" and versions["keras"] == "2.15.0"
    RESULTS["runtime"] = "PASS_METADATA_PROBE; ABI checked only when modules actually run"
    show({**runtime, "HDF5_READY": HDF5_READY, "CNN_READY": CNN_READY})
HDF5_READY = globals().get("HDF5_READY", False)
CNN_READY = globals().get("CNN_READY", False)


### 3.2 Nguồn dữ liệu

Nguồn nghiên cứu là **Edge-IIoTset**, Kaggle revision v5 [3]. Dataset đã được
operator tải local; notebook không tải lại toàn bộ bundle, không upload raw
data lên GitHub và không tự giả định giấy phép Kaggle/metadata là hoàn toàn
nhất quán. Cần giữ source/version/file SHA và điều kiện sử dụng khi làm thí nghiệm.

ML/DNN CSV là curated tabular rows; raw PCAP mới chứa packet sequence để xây
Flow CNN. Không reshape 61 tabular features thành 11 packet features. Bốn DDoS
scenario trong scope là HTTP, TCP SYN, ICMP và UDP; traffic khác không mặc định
benign. Packet labels còn cần IP roles/intervals/capture evidence và policy.

**Cell tiếp theo:** nạp fixture JSON nhỏ từ configs, kiểm literal capture hash và
source-order digest; chỉ kiểm sự tồn tại của các data paths tham khảo. Không đọc
CSV/PCAP thật, không quét dataset hoặc tạo thống kê dataset giả. Clone GitHub mới
không chứa raw data/artifacts là hành vi đúng của `.gitignore`.


In [ ]:
fixture = None
if WORKSPACE is None:
    show({"status": "BLOCKED_SOURCE_NOT_AVAILABLE"})
else:
    from nids_ml.cli.ordering import read_fixture
    from nids_ml.ordering.contracts import source_order_digest
    fixture_path = WORKSPACE / "configs/packet-ordering-synthetic.example.json"
    fixture = read_fixture(fixture_path)
    assert source_order_digest(fixture["packets"]) == fixture["source"]["expected_source_order_sha256"]
    assert hashlib.sha256(fixture["source"]["synthetic_capture_text"].encode()).hexdigest() == fixture["source"]["capture"]["capture_sha256"]
    fixture_before = hashlib.sha256(json.dumps(fixture, sort_keys=True).encode()).hexdigest()
    RESULTS["fixture"] = "PASS_SYNTHETIC_SOURCE_HASH"
    show({"status": RESULTS["fixture"], "input_kind": fixture["kind"], "packets": len(fixture["packets"]),
          "capture": fixture["source"]["capture"], "real_packet_labels": "NOT_PROVIDED",
          "local_data_paths_exist_only": {
              name: (WORKSPACE / path).exists() for name, path in {
                  "ML_CSV": "data/raw/edge-iiotset/kaggle-v5/ML-EdgeIIoT-dataset.csv",
                  "full_dataset_directory": "data/raw/Edge-IIoTset dataset"}.items()},
          "real_data_content_read": False})


#### Kiểm tra nhanh fixture vừa nạp

Cell tiếp theo in các trường quan sát có ích để kiểm chronology: original
ordinal, timestamp và protocol. Missing-field counts được tính trên fixture
synthetic này, không trên Edge-IIoTset. Timestamp regression là trường hợp test
cố ý để kiểm stable processing view; không được sửa source timestamps cho đẹp.


In [ ]:
if fixture is not None:
    packets = fixture["packets"]
    fields = ["ordinal", "timestamp_ns", "ip_protocol", "ip_length"]
    preview = [{key: packet.get(key) for key in fields} for packet in packets]
    regressions = sum(b["timestamp_ns"] < a["timestamp_ns"] for a, b in zip(packets, packets[1:]))
    show({"scope": "SYNTHETIC_FIXTURE_ONLY", "preview": preview,
          "missing_required_fields": {key: sum(packet.get(key) is None for packet in packets) for key in fields},
          "timestamp_regressions": regressions,
          "protocol_counts": dict(collections.Counter(str(packet["ip_protocol"]) for packet in packets)),
          "attack_class_distribution": "NOT_AVAILABLE_WITHOUT_VERIFIED_LABELS"})
else:
    show({"status": "BLOCKED_FIXTURE_NOT_LOADED"})


### 3.3 Khám phá dữ liệu (EDA)

**Mục đích:** trước khi train cần hiểu missing/NaN/Inf, duplicates, class balance,
feature distribution và leakage candidates. Với dữ liệu thật, phải phân biệt
full-scan counts với sampling distributions/correlations; đọc reports EDA có
provenance và checksum, không lấy số từ fixture thay cho dataset.

Cell dưới minh họa timestamp theo capture ordinal và độ dài IP packet của
fixture. Nếu có Matplotlib trong kernel, vẽ hai biểu đồ; nếu không, in trạng
thái visualisation skipped, không tự cài. Đồ thị chỉ phục vụ kiểm phần mềm,
không chứng minh packet nào là DDoS hoặc benign.


In [ ]:
if fixture is not None and importlib.util.find_spec("matplotlib") is not None:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(11, 3))
    axes[0].plot([p["ordinal"] for p in packets], [p["timestamp_ns"] / 1e9 for p in packets], marker="o")
    axes[0].set(xlabel="Original packet ordinal", ylabel="Timestamp (s)", title="Synthetic capture order")
    axes[1].hist([p["ip_length"] for p in packets], bins=3)
    axes[1].set(xlabel="IP length (bytes)", ylabel="Packets", title="Synthetic fixture only")
    fig.tight_layout()
    plt.show()
    plt.close(fig)
    RESULTS["visualisation"] = "PASS_SYNTHETIC_PLOTS"
else:
    RESULTS["visualisation"] = "SKIPPED_SOURCE_OR_MATPLOTLIB_UNAVAILABLE"
show({"status": RESULTS["visualisation"], "dataset_EDA": "NOT_RUN_IN_THIS_NOTEBOOK"})


**Nhận xét từ EDA**

Fixture cố ý chứa timestamp không monotonic theo capture ordinal. Giải pháp là
processing view với thứ tự `(timestamp_ns, source_ordinal)`, không đổi timestamp
hoặc đánh số lại nguồn. Các packet trong fixture có cùng TCP flow và độ dài
tương tự, nên không có cơ sở dùng histogram này để suy class imbalance hoặc
khả năng tổng quát hóa của mô hình.

EDA thật trước đây có reports riêng ngoài repo ML; chúng không được nhúng thành
outputs giả khi người đọc chưa có artifacts. Để hoàn thiện bản nộp, nhóm phải
cung cấp thống kê/biểu đồ thật trên subset/version được chọn, nêu rõ population
và giải thích các vấn đề schema/clock của dữ liệu. Không tự điền counts chưa đo.


## 4. Tiền xử lý dữ liệu

### 4.1 Làm sạch dữ liệu

Trong Flow pipeline, “làm sạch” không có nghĩa là drop packet lỗi hoặc xóa
duplicates âm thầm. Cần xác định eligibility, missing fields, fragments,
unsupported protocols và provenance. Raw giữ nguyên; mỗi biến đổi tạo artifact
mới có version, counts và digest. Duplicate handling khi split phải có policy,
không sửa nguồn để đáp ứng model.

**Cell tiếp theo:** tạo stable ordering view SQLite mới cho fixture; giữ ordinal,
timestamp/hash nguồn và thêm processing index. View được kiểm trước khi dùng.
Output ở run directory mới, không overwrite hoặc cleanup run trước. Đây là
synthetic ordering đã kiểm, không approval để tự sort/gán nhãn raw PCAP thật.


In [ ]:
processing_view = None
references = []
if fixture is not None:
    from nids_ml.ordering.view import build_processing_view, iter_processing_view
    run_parent = Path(os.environ.get("NIDS_NOTEBOOK_RUN_ROOT", str(WORKSPACE / "artifacts/runs"))).resolve()
    assert run_parent.is_relative_to((WORKSPACE / "artifacts/runs").resolve()), "Run parent must stay under artifacts/runs"
    assert run_parent.is_dir(), "Prepare artifacts/runs (or a fresh test parent) first"
    RUN_DIR = run_parent / ("nt120-" + dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dt%H%M%Sz") + "-" + uuid.uuid4().hex[:8])
    RUN_DIR.mkdir(exist_ok=False)
    processing_view = build_processing_view(iter(fixture["packets"]), fixture["source"], RUN_DIR / "view")
    references = list(iter_processing_view(RUN_DIR / "view"))
    assert len(references) == len(fixture["packets"])
    assert [r.processing_index for r in references] == list(range(1, len(references) + 1))
    assert sorted(r.source_packet_ordinal for r in references) == sorted(p["ordinal"] for p in fixture["packets"])
    assert fixture_before == hashlib.sha256(json.dumps(fixture, sort_keys=True).encode()).hexdigest()
    RESULTS["ordering"] = "PASS_SYNTHETIC_STABLE_VIEW"
    show({"status": RESULTS["ordering"], "run_dir": RUN_DIR,
          "processing_order": [r.to_dict() for r in references],
          "processing_order_sha256": processing_view["processing_order_sha256"], "raw_mutation": False})
else:
    RESULTS["ordering"] = "BLOCKED_SOURCE_NOT_AVAILABLE"
    show({"status": RESULTS["ordering"]})


### 4.2 Biến đổi đặc trưng

**Cell tiếp theo:** gọi `OrderedFlowExtractor`, không viết lại LUCID trong notebook.
Flow được nhóm bidirectional theo 5-tuple; capture/run/segment vẫn tách identity.
Window anchor dùng packet đầu và chỉ mở window mới khi `timestamp > anchor+10s`.
Packet đúng biên vẫn thuộc window cũ. Giữ n packet đầu mỗi flow/window; packet
overflow vẫn được đếm/hash để không đánh đồng retained fragment với full window.

| Kênh | Feature | Ý nghĩa |
|---:|---|---|
| 0 | Time | Offset thời gian từ packet đầu fragment |
| 1 | Packet Length | IPv4 length |
| 2 | Higher Protocol | Encoding highest layer được version hóa |
| 3 | IP Flags | Cờ IPv4 |
| 4 | Protocols | Presence bits từ protocol stack/vocabulary cố định |
| 5–8 | TCP Length/Ack/Flags/Window Size | Trường TCP; non-TCP dùng zero theo contract |
| 9 | UDP Length | Trường UDP |
| 10 | ICMP Type | Trường ICMP; zero không tự có nghĩa padding |

Normalization là fixed nominal bounds của implementation, không fit StandardScaler
trên toàn dataset. Packet mask phân biệt real packets với padding; thêm zero rows
sau normalization. Shape n10/n20 phải khớp profile. Inputs không gồm IP/ports,
labels, file names hoặc identities; metadata audit được lưu riêng.


In [ ]:
if processing_view is not None:
    from nids_ml.flows.ordered import OrderedFlowExtractor, OrderedFlowTensor, OrderedExtractionComplete
    from nids_ml.flows.ordered_contracts import configuration as extraction_config
    from nids_ml.ordering.lookup import SyntheticPacketLookup
    for profile in ("flow-cnn-n20-t10", "paper-table3-n10"):
        lookup = SyntheticPacketLookup(fixture["packets"], fixture["source"])
        engine = OrderedFlowExtractor(processing_view, extraction_config(profile))
        events = list(engine.run(iter_processing_view(RUN_DIR / "view"), lookup, input_complete=True))
        assert isinstance(events[-1], OrderedExtractionComplete)
        tensors = [event for event in events if isinstance(event, OrderedFlowTensor)]
        for tensor in tensors:
            values = struct.unpack("<" + "f" * (len(tensor.tensor_bytes) // 4), tensor.tensor_bytes)
            assert all(math.isfinite(value) and 0 <= value <= 1 for value in values)
            real = tensor.metadata["retained_packet_count"]
            assert not any(values[real * 11:]), "Padding must remain zero"
            assert tensor.metadata["supervised_eligible"] is False
        ordered[profile] = tensors
        show({"profile": profile, "stats": engine.stats, "flows": [tensor.to_dict() for tensor in tensors]})
    RESULTS["flow"] = "PASS_SYNTHETIC_N20_N10"
else:
    RESULTS["flow"] = "BLOCKED_PROCESSING_VIEW_MISSING"
show({"status": RESULTS["flow"], "label_approval": "BLOCKED_FOR_REAL_DATA"})


#### 4.2.1 Lưu và đọc lại ordered-HDF5

HDF5 chứa X, metadata/lineage, label states và receipts/completion. Writer chỉ
công bố file khi toàn stream được kiểm, không chỉ khi đã có tensor đầu tiên.
Cell gọi CLI bằng worker đã pin rồi đọc batches để kiểm SHA tensor roundtrip.
Không serialize pickle, không thay UNKNOWN/MIXED thành class 0, không tạo
training data từ fixture. `supervised_only=True` phải trả zero batches.

Nếu worker sai Python/dependency versions, cell in `BLOCKED_RUNTIME` và không tạo
HDF5. Khi đúng runtime nhưng ABI hoặc CLI lỗi, cell báo exception. Bằng chứng
Colab chỉ có khi thực sự chạy trên Colab; local PASS không tự thành Colab PASS.


In [ ]:
if RUN_DIR is not None and HDF5_READY:
    for profile, tensors in ordered.items():
        target = RUN_DIR / (profile + ".hdf5")
        report = run_project([str(WORKSPACE / "scripts/ordered_flow_hdf5.py"), "export-synthetic",
            "--fixture", str(fixture_path), "--view", str(RUN_DIR / "view"),
            "--profile", profile, "--output", str(target)])
        reader_probe = r"""
import hashlib, json, sys
from nids_ml.storage.ordered_hdf5 import iter_batches, validate_hdf5
path=json.load(sys.stdin)['path']; report=validate_hdf5(path)
hashes=[]; metadata=[]
for batch in iter_batches(path):
    assert batch['y'].tolist()==[-1]*len(batch['X']) and not batch['supervised_mask'].any()
    hashes.extend(hashlib.sha256(t.tobytes()).hexdigest() for t in batch['X'])
    metadata.extend(batch['metadata'])
assert list(iter_batches(path,supervised_only=True))==[]
print(json.dumps(dict(report=report,tensor_sha256=hashes,metadata=metadata)))
"""
        checked = run_project(["-c", reader_probe], {"path": str(target)})
        assert checked["tensor_sha256"] == [hashlib.sha256(t.tensor_bytes).hexdigest() for t in tensors]
        assert checked["metadata"] == [t.metadata for t in tensors]
        assert checked["report"]["content_sha256"] == report["content_sha256"]
        hdf5_reports[profile] = report
        show({"profile": profile, "status": report["status"], "shape": report["shape"],
              "label_counts": report["label_counts"], "supervised_count": report["supervised_count"],
              "file_sha256": report["file_sha256"], "tensor_roundtrip": "EXACT_MATCH"})
    RESULTS["hdf5"] = "PASS_SYNTHETIC_ROUNDTRIP"
else:
    RESULTS["hdf5"] = "BLOCKED_RUNTIME_OR_SOURCE"
show({"status": RESULTS["hdf5"]})


### 4.3 Chia tập huấn luyện, kiểm định và kiểm tra

Không dùng `train_test_split` ngẫu nhiên trên từng packet/flow window khi các
mẫu cùng capture/session có thể đi sang nhiều tập. Cần chọn split unit và group
constraints trước; giữ metadata để kiểm overlap, duplicates và temporal leakage.
Validation dùng chọn cấu hình; test được giữ kín đến bước đánh giá cuối.

**Cell tiếp theo chỉ tạo split plan**, đếm samples hiện có và kết luận eligibility.
Không sinh indices hoặc giả lập train/val/test bằng hai tensor UNKNOWN. Tỷ lệ
split, capture allocation và cách chia clients chưa được duyệt thì để UNKNOWN.


In [ ]:
split_plan = {
    "status": "BLOCKED_VERIFIED_GROUND_TRUTH_AND_SPLIT_APPROVAL",
    "available_demo_flows": {name: len(tensors) for name, tensors in ordered.items()},
    "supervised_eligible_demo_flows": sum(t.metadata["supervised_eligible"] for tensors in ordered.values() for t in tensors),
    "proposed_split_unit": "CAPTURE_OR_APPROVED_SESSION_GROUP; not individual overlapping windows",
    "train_validation_test_ratio": None,
    "required_checks": ["verified labels", "capture/group disjointness", "duplicate and temporal leakage",
                        "feature/normalization version", "frozen test set", "client partition provenance"],
    "indices_created": False,
}
assert split_plan["supervised_eligible_demo_flows"] == 0
RESULTS["split"] = split_plan["status"]
show(split_plan)


### 4.4 Xử lý mất cân bằng nhãn và nhãn không xác định

Với dataset thật, đo phân bố lớp trước, rồi cân nhắc class weights hoặc sampling
**chỉ trên train**. Không rebalance validation/test hoặc lấy accuracy làm độ đo
duy nhất. UNKNOWN không phải benign; MIXED không được majority vote thành nhãn 0/1.

Cell minh họa strict label aggregation bằng giả thiết synthetic độc lập, không
áp vào Edge-IIoTset hoặc source ordinals đã được reorder. Resolver hiện yêu cầu
identity/order contract riêng và n20; không tự renumber để vượt check. Ví dụ
20 benign + 1 DDoS phải giữ MIXED cho full window dù retained 20 packet đều benign.
Đây là kiểm logic, không phải evidence xác nhận nhãn thật.


In [ ]:
if WORKSPACE is not None:
    from nids_ml.labeling.resolver import Label, PacketDecision, aggregate_window
    cases = {
        "all_benign_assumed_synthetic": [Label.VERIFIED_BENIGN] * 3,
        "all_ddos_assumed_synthetic": [Label.VERIFIED_DDOS] * 3,
        "unknown": [Label.UNKNOWN] * 3,
        "mixed": [Label.VERIFIED_BENIGN, Label.VERIFIED_DDOS],
        "positive_after_retained_n20": [Label.VERIFIED_BENIGN] * 20 + [Label.VERIFIED_DDOS],
    }
    label_results = {}
    for name, labels in cases.items():
        decisions = [PacketDecision(label, ("SYNTHETIC_ASSUMPTION_ONLY",),
            identity=("synthetic-label-demo", "segment-1", "run-1", "flow-a"), ordinal=i, timestamp_ns=i)
            for i, label in enumerate(labels, 1)]
        label_results[name] = aggregate_window(decisions, full_window_complete=True).to_dict()
    assert label_results["unknown"]["status"] == "QUARANTINE"
    assert label_results["positive_after_retained_n20"]["full_window_label"] == "MIXED"
    assert label_results["positive_after_retained_n20"]["status"] == "QUARANTINE"
    RESULTS["label_safety"] = "PASS_SYNTHETIC_AGGREGATION; no real approvals"
    show(label_results)
else:
    RESULTS["label_safety"] = "BLOCKED_SOURCE_NOT_AVAILABLE"
    show({"status": RESULTS["label_safety"]})


## 5. Phương pháp tiếp cận

### 5.1 Giải thuật hoặc mô hình được lựa chọn

Flow CNN nhỏ: Input `(n,11,1)` → Conv2D 64 filters, kernel `(3,11)` → ReLU →
GlobalMaxPooling2D → Flatten → Dense 1 sigmoid. Conv có 2.176 và Dense có 65
tham số, tổng 2.241. N10 cho output Conv `(8,1,64)`; n20 cho `(18,1,64)`.

Lý do chọn: model nhỏ phù hợp nghiên cứu CPU/edge và khớp phần Flow của bài
tham khảo. Số tham số nhỏ chưa chứng minh inference nhanh hoặc accuracy tốt;
cần benchmark trên hardware và split thật. Không dùng số 25.775 tham số của
Packet branch cho Flow model này.


### 5.2 Pipeline tổng thể

```text
PCAP + capture provenance + verified annotations (chưa đủ gate)
    → decoder/eligibility → approved processing order → label resolver
    → flow tensors + lineage → HDF5 → leakage-safe split
    → centralized CNN baseline → validation/model selection → final test
    → CNN User-TF adapter → Swarm rounds trên hai edge nodes (bước sau)
```

Nhánh chạy được trong notebook hiện tại:

```text
Synthetic fixture → stable view → ordered tensors n20/n10
    → HDF5 roundtrip nếu đúng runtime → untrained CNN forward nếu đúng runtime
```

SN giữ registry/round coordination; SL trao đổi updates và chọn aggregator
deterministic theo round cho sample-weighted FedAvg. Kiến trúc đó không tự
chứng minh decentralized aggregation hoặc CNN integration đã hoàn tất.


### 5.3 Cài đặt mô hình và kiểm tra forward

Cell gọi model source đã có, kiểm model summary/parameter counts và chạy forward
trên tensors synthetic vừa tạo. Worker CPU, compile_model=false: không optimizer
update, không fit. Predictions chỉ là xác suất từ model **chưa được train**;
không diễn giải chúng là detector hoặc dùng để tính accuracy của đồ án.

Nếu worker khác contract, kết quả là BLOCKED_RUNTIME. Không tự nâng TensorFlow
để cho notebook chạy rồi gắn kết quả vào phiên bản ARM64 đã kiểm.


In [ ]:
model_reports = {}
if WORKSPACE is not None:
    from nids_ml.models.flow_cnn import configuration as model_config
    show({"configured_models": {name: model_config(name, seed=SEED) for name in ("flow-cnn-n20-t10", "paper-table3-n10")},
          "configured_parameter_count": (3 * 11 * 1 + 1) * 64 + (64 + 1) * 1,
          "configuration_is_not_runtime_execution": True})
if ENABLE_CNN_FORWARD and CNN_READY and ordered:
    model_probe = r"""
import json, sys
import numpy as np
from nids_ml.models.flow_cnn import configuration, build_model, inspect_model, predict_flow, runtime_info
value=json.load(sys.stdin); config=configuration(value['profile'],seed=value['seed'])
X=np.stack([np.frombuffer(bytes.fromhex(t['hex']),dtype='<f4').reshape(t['shape']) for t in value['tensors']])
model=build_model(config,compile_model=False)
prediction=predict_flow(model,X,config)
print(json.dumps(dict(model=inspect_model(model,config),runtime=runtime_info(),
                     predictions_untrained=prediction.tolist(),input_shape=list(X.shape),training=False)))
"""
    for profile, tensors in ordered.items():
        payload = {"profile": profile, "seed": SEED,
                   "tensors": [{"shape": list(t.shape), "hex": t.tensor_bytes.hex()} for t in tensors]}
        report = run_project(["-c", model_probe], payload)
        assert report["model"]["trainable_parameters"] == 2241 and report["training"] is False
        model_reports[profile] = report
        show({"profile": profile, **report})
    RESULTS["cnn_forward"] = "PASS_UNTRAINED_SYNTHETIC_FORWARD"
else:
    RESULTS["cnn_forward"] = "BLOCKED_RUNTIME_OR_SOURCE" if ENABLE_CNN_FORWARD else "SKIPPED_BY_CONFIGURATION"
show({"status": RESULTS["cnn_forward"], "supervised_training": "BLOCKED"})


### 5.4 Siêu tham số và phương án training sau khi mở gate

Table 12 của [1] nêu Adam, learning rate 0,001, batch size 128, 10 epochs và
binary crossentropy trong setup multimodal. Áp các giá trị đó cho Flow-only là
lựa chọn implementation cần công bố; Adam beta/epsilon/seed/initializer không
được tự ghi VERIFIED_FROM_PAPER. Các giá trị đề xuất nằm trong source config.

Cell xuất training reference và các điều kiện để chạy. Không tuning/test-set
feedback; không gọi `model.fit` hoặc thay UNKNOWN thành nhãn để mở gate. Dòng
“No. of System Frequencies=1024” trong bài còn thiếu semantics, không tự chuyển
thành số rounds, Hz hoặc sync frequency của hệ thống hiện tại.


In [ ]:
if WORKSPACE is not None:
    from nids_ml.models.flow_cnn import training_reference
    reference = training_reference()
else:
    reference = None
training_plan = {
    "status": "BLOCKED_REAL_TRAINING_NOT_IMPLEMENTED_IN_NOTEBOOK",
    "reference_configuration": reference,
    "prerequisites": ["approved packet/flow ground truth", "complete real flow export",
                      "approved leakage-safe split", "frozen preprocessing/feature schema",
                      "runtime validation", "experiment budget and baseline protocol"],
    "fit_called": False, "optimizer_updates": 0, "test_set_used_for_tuning": False,
}
show(training_plan)


## 6. Thực nghiệm và đánh giá kết quả

### 6.1 Thiết lập thực nghiệm

Phần thực sự chạy trong notebook dùng seed 42, fixture nhỏ, thread limits=1 cho
worker, CPU-only forward và run path mới. Versions/shape/counts/hashes được in
từ lệnh đã chạy, không từ ảnh cũ. Kernel metadata hoặc worker.machine không tự
chứng minh một bài native two-node Swarm run.

Khi làm dữ liệu thật, phải bổ sung dataset version/files SHA, capture/annotation
provenance, split IDs/sizes, client sample counts, epochs/rounds, resource budget
và version/commit source. Không suy CNN results từ tiny-regressor FedAvg đã kiểm.


### 6.2 Độ đo đánh giá

Với positive=1 DDoS: precision=TP/(TP+FP), recall=TP/(TP+FN), F1 là trung bình
điều hòa precision/recall; thêm confusion matrix và ROC-AUC khi test có đủ hai
lớp. Accuracy đơn lẻ có thể che class imbalance. Threshold phải được quyết định
trên validation, không chọn theo test. Test evaluation chỉ sau model selection.

Software validation dùng phép đo khác: số packet/flow, shape/dtype, padding,
digest consistency, numeric/metadata roundtrip và completion safety. Những phép
kiểm này không thay precision/recall/F1 trên dữ liệu thật.


### 6.3 Kết quả định lượng

Cell tạo bảng kết quả từ state thực tế của lần Run All. Metrics chưa đo để
`None`, không điền zero hoặc số trong bài báo. HDF5 được tạo từ fixture có
supervised_count=0; predictions của model chưa train không có giá trị đánh giá
detector. Nếu runtime thiếu, bảng phải ghi BLOCKED thay vì khoe kết quả cũ.


In [ ]:
software_results = []
for profile, tensors in ordered.items():
    software_results.append({
        "profile": profile, "synthetic_packets": len(fixture["packets"]), "flow_tensors": len(tensors),
        "tensor_shape": list(tensors[0].shape) if tensors else None,
        "HDF5_status": hdf5_reports.get(profile, {}).get("status", "BLOCKED_RUNTIME"),
        "supervised_count": hdf5_reports.get(profile, {}).get("supervised_count"),
        "CNN_forward": "PASS_UNTRAINED" if profile in model_reports else "NOT_RUN",
    })
real_metrics = {key: None for key in ("accuracy", "precision", "recall", "f1", "roc_auc", "confusion_matrix")}
show({"software_validation": software_results, "real_dataset_test_metrics": real_metrics,
      "reason": "No approved real train/validation/test artifacts or trained CNN", "training": False})


### 6.4 So sánh với baseline hoặc phương pháp khác

Đề xuất baseline đơn giản và centralized Flow CNN trên **cùng split/feature
contract** trước khi so sánh Swarm CNN. Với labels verified mới tính được
majority-class baseline; không lấy UNKNOWN làm majority benign. So sánh n10/n20
ở notebook hiện tại chỉ kiểm representation/shape, không so sánh F1.

Cell dưới tạo comparison plan, để scores chưa có ở `None`. BaselinePaper là
source legacy đã review, không phải kết quả baseline được notebook tự xác nhận
và không được import để tự áp folder labels cho Edge-IIoTset.


In [ ]:
comparison_plan = [
    {"method": "majority-class baseline", "test_f1": None, "status": "BLOCKED_VERIFIED_LABELS_AND_SPLIT"},
    {"method": "centralized Flow CNN n20", "test_f1": None, "status": "NOT_TRAINED"},
    {"method": "centralized Flow CNN n10", "test_f1": None, "status": "NOT_TRAINED"},
    {"method": "two-node Swarm Flow CNN", "test_f1": None, "status": "CNN_ADAPTER_AND_REAL_TRAINING_NOT_IMPLEMENTED"},
]
assert all(row["test_f1"] is None for row in comparison_plan)
show(comparison_plan)


## 7. Phân tích và thảo luận

### 7.1 Phân tích kết quả

Nếu ordering/flow/HDF5 PASS, có thể kết luận phần mềm giữ source identity,
processing order và numeric tensors xuyên qua storage trên fixture này. N20/n10
cho kích thước khác nhau nhưng cùng 11 features và kiến trúc CNN 2.241 params.
Hai flow windows của fixture không đại diện distribution của Edge-IIoTset.

Forward PASS xác nhận model chấp nhận representation trên đúng runtime, không
chứng minh model phân biệt tấn công. Không dùng predictions ngẫu nhiên để giải
thích feature importance. Kết quả flow retention/overflow cần xem full-window
metadata, không chỉ tensor đã cắt n packet.


### 7.2 Kiểm tra độ tin cậy

- **Overfitting:** chưa có đường train/validation loss thật để kết luận. Khi có,
  so sánh learning curves trên split độc lập, không đưa test vào early stopping.
- **Leakage:** IP/ports/capture IDs/labels ở metadata ngoài X; vẫn phải kiểm group
  overlap, duplicated flows và temporal/scenario fingerprints khi chia tập thật.
- **Imbalance:** đo class distribution thật; không lấy histogram fixture suy ra.
- **Lineage:** hash chứng minh consistency của artifact, không xác nhận annotation
  là đúng hoặc source đã được cấp phép.
- **Runtime:** local Windows/ARM tests của bundle cũ không thay live Colab hoặc
  native validation cho source mới. Không suy multi-host training từ notebook.


### 7.3 Hạn chế

1. Verified packet/flow ground truth và global mapping còn thiếu; real labeling,
   train/validation/test artifacts và training bị chặn.
2. Một số capture thật có timestamp regressions; một số CSV có clock/schema
   issues. Synthetic processing view không tự giải quyết provenance/label policy.
3. Notebook chưa được chạy trên Google Colab thực tế; default Colab versions có
   thể khác pins. Source/dependencies phải có trên runtime để demo đầy đủ.
4. Chưa CNN adapter/Swarm training; authentication/TLS và production threat model
   nằm ngoài MVP. Không dùng Colab để thay thế native two-board validation.
5. Model reproduction là adaptation có công bố assumptions, không khẳng định
   tái lập nguyên multimodal HPE/blockchain hoặc F1 của tác giả.


## 8. Kết luận và hướng phát triển

### 8.1 Tóm tắt đóng góp hiện tại

Đã tổ chức source theo data/EDA/PCAP/flows/ordering/labels/storage/models, có
versioned contracts và synthetic tests. Notebook trình bày được công việc từng
bước, gọi implementation hiện có và có assertions cho source hash, ordering,
tensor/padding, HDF5 integrity và CNN forward khi đúng runtime.

Đóng góp ở giai đoạn này là pipeline và khả năng kiểm chứng. Chưa có kết luận
về accuracy/F1 phát hiện DDoS thật hoặc lợi ích của Swarm CNN. Bảng cuối notebook
phải được xem cùng các trạng thái BLOCKED, không chỉ thông báo cell đã chạy.


### 8.2 Hướng phát triển

1. Hoàn tất annotation/provenance và label/order approvals cho dữ liệu thật.
2. Export real flow HDF5 trên data local, tạo split có group/temporal safeguards.
3. Train centralized CNN/baseline, chọn hyperparameters bằng validation, final
   evaluation bằng test đã đóng băng; lưu configs/checkpoints/metrics thật.
4. Viết CNN training adapter cho Independent User-TF, test weight contracts và
   hai rounds; sau đó kiểm native trên Raspberry Pi 5 và Orange Pi 5 Pro.
5. Hoàn tất Colab compatibility và notebook Run All với môi trường được pin,
   source commit/data contracts rõ ràng trước khi nộp giảng viên.

Các bước này là kế hoạch, không được tự chạy chỉ vì notebook có mục tương ứng.


## 9. Đóng góp của các thành viên

Người dùng tự điền thông tin thật; bảng này phải khớp bảng ở đầu notebook.
Không tự ghi tất cả thành viên “tham gia đầy đủ” khi chưa có xác nhận.

| MSSV | Họ tên | Phần việc cụ thể và evidence |
|---|---|---|
| [TỰ ĐIỀN] | [TỰ ĐIỀN] | [TỰ ĐIỀN] |


## 10. Tài liệu tham khảo

[1] *A Multimodal Swarm Learning Approach for DDoS Detection in Internet of Things
Infrastructure*, DOI: [10.4108/eetinis.131.9961](https://doi.org/10.4108/eetinis.131.9961).
Table 3/12 và phần flow preprocessing được đối chiếu ở ARM-011A; không dùng
scores của bài báo làm kết quả nhóm.

[2] LUCID, DOI: [10.1109/TNSM.2020.2971776](https://doi.org/10.1109/TNSM.2020.2971776);
[reference source](https://github.com/doriguzzi/lucid-ddos/tree/960ef782efae9f0535e4eef0c6d83edd311cda23).
Đây là revision source đã audit, chưa chứng minh authors của [1] dùng cùng commit.

[3] [Edge-IIoTset — Kaggle v5](https://www.kaggle.com/datasets/mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot/versions/5).
Giữ dataset source/version và license differences trong data-source audit;
notebook không tái phân phối dataset hoặc tự tải lại.

[4] [TensorFlow API](https://www.tensorflow.org/api_docs/python/tf/keras).
Implementation hiện pin TF 2.15.1/Keras 2.15.0, không tự áp API mới chưa regression.

[5] [h5py documentation](https://docs.h5py.org/en/stable/).
Runtime hiện pin h5py 3.16.0/NumPy 1.26.4; HDF5 chỉ lưu numeric tensors và bounded JSON.

[6] [Colab FAQ](https://research.google.com/colaboratory/faq.html) và
[runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html).
Môi trường thay đổi; xem probe thực tế thay vì giả định Python/GPU availability.

Source chính trong `src/nids_ml/`; configs/contracts/tests trong cùng repo.
Task reports/evidence ở workspace phát triển bên ngoài repo ML; notebook không
phụ thuộc chúng để chạy demo synthetic. Clone mới không chứa raw/runtime outputs.


## 11. Phụ lục (tùy chọn)

### 11.1 Assertions và bảng gate của lần Run All hiện tại

Cell cuối kiểm các invariants đã được thực thi và in trạng thái rõ ràng. Không
đọc log cũ để giả lập một test mới. Không import chạy toàn bộ training suite
trong notebook. Nhóm có thể chạy riêng các tests synthetic ordering/storage
theo README khi cần regression; tests PASS không phải dataset accuracy.

Run path là nơi operator xem outputs thực tế. Notebook không xóa artifacts cũ
và không upload logs/data. Trước commit, bỏ cell outputs; evidence chạy thật cần
lưu riêng có runtime/source/data identity. `PUBLIC_RELEASE`, physical Swarm và
Colab runtime verification không được tự đổi trạng thái bởi cell này.


In [ ]:
if ordered:
    for profile, tensors in ordered.items():
        n = 20 if profile == "flow-cnn-n20-t10" else 10
        assert all(t.shape == (n, 11, 1) for t in tensors)
        assert sum(t.metadata["real_packet_count"] for t in tensors) == len(fixture["packets"])
        assert all(t.metadata["full_window_label"] == "UNKNOWN" for t in tensors)
    RESULTS["notebook_assertions"] = "PASS_SYNTHETIC_INVARIANTS"
else:
    RESULTS["notebook_assertions"] = "BLOCKED_SOURCE_NOT_AVAILABLE"

SUMMARY = {"execution_scope": "SYNTHETIC_DEMO_AND_OPTIONAL_UNTRAINED_FORWARD", "gates": RESULTS,
           "run_directory": str(RUN_DIR) if RUN_DIR else None,
           "real_dataset_training": "BLOCKED", "submission_readiness": "BLOCKED_PENDING_REAL_EXPERIMENTS",
           "live_Colab_validation": "NOT_VERIFIED", "CNN_native_two_node_training": "NOT_VERIFIED",
           "no_Docker_SSH_or_Git_writes": True, "optimizer_updates": 0}
show(SUMMARY)


## 12. Checklist tự đánh giá trước khi nộp

Các checkboxes dưới là kiểm soát chất lượng bản nộp, không được tự tick toàn bộ
vì synthetic cells chạy được. Người dùng cập nhật sau khi có evidence thật.

- [x] Bài toán có input/output/scope và positive class rõ ràng.
- [x] Giữ nguyên cấu trúc 13 mục của mẫu; code/Markdown giải thích từng bước.
- [x] Phân biệt synthetic validation với dataset training và physical Swarm.
- [x] Có seed, version gates, source fingerprint và artifact/lineage checks.
- [ ] Điền thông tin nhóm/đóng góp và đổi tên NhomXX.
- [ ] Source repo/commit được công bố và lấy lại được trên runtime mới.
- [ ] Dataset provenance/verified annotations và label/order policy được khép.
- [ ] EDA thật có biểu đồ/nhận xét và phạm vi full-scan/sampling rõ ràng.
- [ ] Split thật tránh leakage; train/validation/test IDs và counts được ghi.
- [ ] Training/baselines/model selection đã chạy và có checkpoint/metrics thật.
- [ ] Final test có confusion matrix, precision/recall/F1 và phân tích lỗi.
- [ ] Có phân tích overfitting/imbalance và limitations dựa kết quả thật.
- [ ] Notebook Run All được xác minh trên Google Colab với mọi bước cần thiết.
- [ ] Bỏ outputs chứa thông tin riêng hoặc dataset trước Git; lưu evidence riêng.
- [ ] Các thành viên thống nhất nội dung, tài liệu tham khảo và số liệu báo cáo.


## 13. Gợi ý chuẩn bị phần trình bày trước lớp

1. Nêu bài toán DDoS/IoT và phân biệt Flow CNN với tabular/Packet branch.
2. Giải thích packet → 5-tuple/window → 11 features → n20/n10 tensor; chỉ vào
   một dòng lineage thật của demo để chứng minh hiểu ordering/padding.
3. Chạy synthetic demo và nói rõ nó kiểm software, không đo detection accuracy.
4. Khi đã đủ dữ liệu/thực nghiệm: trình bày EDA thật, split safeguards, learning
   curves, confusion matrix và baseline comparison; không dùng số của tác giả.
5. Trình bày physical two-node Swarm evidence ở phần riêng; chỉ gọi là CNN Swarm
   khi có CNN integration/training evidence, không suy từ tiny-model FedAvg.
6. Kết thúc bằng limitations/blockers và phần việc nhóm thực sự đã làm.

Chuẩn bị câu trả lời: vì sao 11 features thay vì 61 CSV columns; n10/n20 khác
nhau thế nào; tại sao unmatched=UNKNOWN; vì sao hash không chứng minh label đúng;
tại sao random window split có thể leakage; Colab khác native ARM64 ở điểm nào.
